# Pymarc

#### Install pymarc in your environment:
1. Select from top menu: **File::New::Terminal**
2. Write: **pip install pymarc**
3. Close the terminal window when finished

In [2]:
import pymarc
from pymarc import Record, Field, Subfield, Indicators, parse_xml_to_array, XMLWriter

**pymarc** is a Python code library with basic functions for inspecting and modifying  Marc records. \
pymarc is documented at [Pymarc documentation](https://pymarc.readthedocs.io/en/stable/)

Examples of what we can do:
1. Read ("parse") a set of Marc records extracted from Alma (in MarcXML)
2. Display major information elements from the records, in a simple way
3. Inspect and modify fields and subfields
4. Write the dataset to file as MarcXML
5. ...and more

**This notebook contains some concrete examples of pymarc usage, for you to try and elaborate further with your dataset.** \
**You can try code in this notebook, or create your own notebook for your code, using this notebook as reference.**

### 1. Read the Marc records from your dataset

In [3]:
#Read the data file into a list of Marc records

#Replace the file path below with the path to your own dataset

myrecords=parse_xml_to_array('datasett/norbokReg2026feb.xml')

In [4]:
#How many records in the set?
len(myrecords)

3441

In [ ]:
marcpy1.fieldCounter(myrecords)

##### Define a couple of small subsets to begin with

In [10]:
#For example, the first and last 10 records in your set
first10=myrecords[0:10]   
last10=myrecords[-10:]

### 2. Display information from your records
By executing the parse_xml_to_array as above, myrecords becomes a list of  Marc-poster. In pymarc, each Marc record exists as a Record object (i.e. an instance of the class Record)\
Check out the [functions and properties defined for Record](https://pymarc.readthedocs.io/en/stable/#module-pymarc.record), and display (by using *print*) a list of selected information from the records in one of your smaller subsets \
**Hint**: Create a for-loop, for each Record fetch the information elements you have chosen to diaplay (e.g. author, title, isbn etc.) \
```
for rec in <list>:
    print(rec.<property1>, rec.<property2>,...)
```

In [7]:
for rec in last10:
    print(rec.author, '\t', rec.title,'\t', 'ISBN:', rec.isbn, 'LDR:', rec.leader)

Dahl, Hanne Hafnor 1970- (NO-TrBIB)9059837 aut 	 Matematikk 2 frå Cappelen Damm 	 ISBN: 9788202891923 LDR: 03471cam a2200505 c 4500
Dahl, Hanne Hafnor 1970- (NO-TrBIB)9059837 aut 	 Matematikk 2A fra Cappelen Damm 	 ISBN: 9788202891893 LDR: 03729cam a2200481 c 4500
Walt Disney Company (NO-TrBIB)90554172 aut 	 Donald som postmann 	 ISBN: 9788242984005 LDR: 00000cim a22     5c 4500
Walt Disney Company (NO-TrBIB)90554172 ccp 	 Trampe 	 ISBN: 9788242983978 LDR: 00000cim a22     5c 4500
None 	 Løvenes konge 	 ISBN: 9788242983954 LDR: 00000cim a22     5c 4500
Walt Disney Company (NO-TrBIB)90554172 ccp 	 Lady og Landstrykeren Fant på eventyr 	 ISBN: 9788242983947 LDR: 00000cim a22     5c 4500
Walt Disney Company (NO-TrBIB)90554172 ccp 	 Den flygende musa 	 ISBN: 9788242983985 LDR: 00000cim a22     5c 4500
Ḥusiynī, Mīr Saʻīd (NO-TrBIB)1784289772763 aut 	 Sāyih'hā-yi bīqarār 	 ISBN: 9788294106097 LDR: 01908nam a2200325 c 4500
Jewell, Lisa 1968- (NO-TrBIB)42559 aut 	 Det kunne vært henne 	 ISBN

**An alternative print statement** is to use a *formatted string literal*, with variable expressions in curly brackets are embedded in plain text. \
This makes it easier to create readable strings with some variable content.
```
print(f"Author and title: {rec.author} {rec.title}, ISBN: {rec.isbn} LDR: {rec.leader}") 
```
#### **DIY: Try it on your own data**

In [9]:
print(f"Author and title: {rec.author} {rec.title}, ISBN: {rec.isbn}, LDR: {rec.leader}") 

Author and title: Hjardar, Espen 1969- (NO-TrBIB)6043017 aut Matematikk 8-10 frå Cappelen Damm, ISBN: 9788202914172, LDR: 03368cam a2200409 c 4500


### 3. Define a function doing the above
We might want to do the above repeatedly, for various datasets. To make it easier, define a function performing the operation, taking a **list of Record objects** as argument. \
Here is a template:
```
def recordsOverview(records):
    #<a short description of what the function does, and/or what it returns, if anything>
    for rec in records:
        print(rec.<property1>, rec.<property2>,...)
```
#### **DIY: Now define *recordsOverview* and run it to display the information**

### 4. Inspecting records, fields and subfields in a more detailed manner
To check (and possibly update) very specific things in the records, we'll need other functions and methods than those used above. \
For example, the property author returns the whole (100/110/111) field as one string, but sometimes we need to focus on specific subfields.\

Some case examples: 
* A specific code within 008 is to be replaced with another code, because of change in cataloguing practice.
* Someone has discovered an erroneous source code in ($2) in the catalogue. Then we need to find and correct all occurrences of the erroneous code.
* A new bibliography or virtual collection is being established, and we need to assign bibliography code (913) to a set of records.
* etc

**To access such detailed information, we need to know the *pymarc data model***

### 5. pymarc's data model - the bare minimum
1. The records you have read and loaded exist in the form of a list of  **Record**-objects
2. Each Record object contains one **leader** and is associated with a set of **Field** objects
3. Each Field object contains a **tag** (field code), and is linked to an object **Indicators** and associated with a set of **Subfield** objects \
**Note:** The indicators and subfields are not really objects (in the sense of class members), but are so called NamedTuple instances. Ignore for now... 

### 6. Accessing Fields of a Record
There a several ways of fetching fields of a record, two of which are mentioned here:

The basic access method for fetching fields from a record rec:
* **rec.get_fields(fieldcode1, fieldcode2,...)** --> returns a **list of all Field objects** with fieldcode equal to any of the codes given as argument

For non-repeatable fields, or if fetching all fields with a certain code is not important, this may be easier:
* **rec.get(fieldcode)** ---> returns the **first Field object** with fieldcode <fieldcode>) \

#### **DIY: Try it out with your data**

### 7. Accessing Fields' contents: Subfields
Check out the documentation of the [access methods of Field objects](https://pymarc.readthedocs.io/en/stable/#module-pymarc.field).

The basic access method for fetching subfield values of a Field fld:
* **fld.get_subfields(subfieldcode1, subfieldcode2,...)** --> returns a **list of all subfield values** with code equal to any of the codes given as argument \
**Note**: The values are listed in the order as given in the data, not in the order given as argument \
Consider the field fld with value
```
700  1\$aEliassen, Ida Sofie Hagen$0(NO-TrBIB)1764845785215$4art
```
Then
```
fld.get_subfields('4', 'a') returns
['Eliassen, Ida Sofie Hagen', 'art']
```
* **fld.get(subfieldcode)** ---> returns the **first subfield value** with code subfieldcode)
* **fld.subfields** returns a list of all the  **Subfield "objects"** of the field \
With the example above: 
```
fld.subfields returns
[Subfield(code='a', value='Eliassen, Ida Sofie Hagen'),
 Subfield(code='0', value='(NO-TrBIB)1764845785215'),
 Subfield(code='4', value='art')]
 ```
 * **fld.subfields_as_dict** returns the subfields as a dict, representing the mapping form subfield code to a list of values
```
fld.subfields_as_dict() returns
{'a': ['Eliassen, Ida Sofie Hagen'],
 '0': ['(NO-TrBIB)1764845785215'],
 '4': ['art']}
```

#### **DIY: Try it out with your data**

### 8. Accessing Fields' contents: Indicators and overall value
Consider the field ***fld*** with value
```
700  1\$aEliassen, Ida Sofie Hagen$0(NO-TrBIB)1764845785215$4art
```
Indicators may be fetched separately, or both at once:
* **fld.indicator1** --> returns the first indicator, a **string of length=1**, e.g. \
  ```
  '1'
  ```
* ditto for **fld.indicator2**
* **fld.indicators** returns both indicators as a NamedTuple, e.g. \
```
Indicators(first='1', second=' ')
```
The overall value of a field - positional as well as other fields - can be fetched all at once, by calling the value() method:
* **fld.value()** returns the value of the field as one string, including all subfield values, but excluding indicators. e.g.
```
'Eliassen, Ida Sofie Hagen (NO-TrBIB)1764845785215 art'
```
* for **positional fields**, their value can also be fetched by **fld.data**. 
#### **DIY: Try it out with your data**

### 9. Changing your records
Your dataset can be updated in numerous ways using pymarc: 
* Updating Record objects by **adding** or **removing** or **changing** Field objects
* Updating Field objects by **adding or deleting Subfields**, changing **indicators** or **tag**
* Creating new Record, Field and Subfield objects \
**Note**: As long as we are processing records extracted from the catalogue, creating new Records are not often necessary.

#### 9.1. Creating a new Field object and adding  it to a record

To create a new **data field**, use the example below as a template:
```
newFld = Field(
    tag = '913',
    indicators = Indicators(' ',' '),
    subfields = [
        Subfield(code='a', value='Littforsk'),
        Subfield(code='b', value='NB'),
    ])
```
To create a new **control field** (position based) is simpler:
```
newCFld = Field(tag='001', data='999923031939002201')
```
Then, newFld can be **added** to a record **rec**, using either **add_field** or **add_ordered_field** methods (the latter **recommended**), like this:
```
rec.add_field(newFld)  --> adds newFld to the end of rec
rec.add_ordered_field(newFld)  --> adds newFld to rec, placing it according its tag's place in the Marc specification
```
#### DIY: Try it on one of your own records. Show the result by printing the updated record!

#### 9.2. Adding subfields in existing Field objects
Subfields may be added to an existing Field object  **thisFld**, by using the **add_subfield** method, like this:
```
thisFld.add_subfield(code='t', value='Gruffalo', pos=2)
```
or just
```
thisFld.add_subfield('t', 'Gruffalo', pos=2)
```
Both of the above **inserts a new subfield into thisFld**, at the place given by **pos**. If no pos given, the subfield is added to the end of the field.
#### DIY: Try it on one of your own Fields. Show the result by printing the updated field or record!


#### 9.3. Removing fields from records and subfields from fields
1. Unwanted Field objects can be removed from its mother record **rec**, by using the **remove_fields** or **remove_field** methods, like this:
```
rec.remove_field(fld1, fld2,...)  --> removes fld1, fld2. etc from the record. 
```
or
```
rec.remove_fields(tag1, tag2, ...) ---> removes all fields with the given field codes from rec
```
**Note:** **remove_field** takes one or more **Field objects** as arguments, while **remove_fields** takes **field codes** (as strings, e.g. '600') as arguments
2. Unwanted subfields can be removed form a Field object **fld** by calling **delete_subfield**, like this:
```
fld.delete_subfield(code) ---> deletes the first occurrence of subfield with code (e.g. 'a') from fld.
```
#### DIY: Try it on one of your own Records. Show the result by printing the updated field or record!


### 10. Writing your updated records to file
To save changes done to your dataset ***myrecords***, write the records to file, using the following as a template
```
    writer = XMLWriter(open(<filename>, 'wb'))
    for rec in <marcrecs>:
        writer.write(rec)
    writer.close()

```
**Note:** To preserve your original dataset, be sure to give another *filename* than that of the file holding your original dataset

#### DIY: Try it on your own Records. Then you can try to read the new file (with parse_xml_to_array)
